# Example 0.9.3 / 1: Solve an Assignment Model Instance

Companion notebook for **0.9.3 The Assignment Model** in the AMPL lab Documentation, section "Solve an Assignment Model Instance". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve an Assignment Model Instance

The AMPL book assigns people to offices by preference rank, where 1 is a person's first choice. Here three of its people and three of its offices keep their ranks. `cost` is a dictionary keyed by `(person, office)` pairs, and the run prints the pairs whose `Assign` is 1:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PEOPLE;                         # People to assign
set JOBS;                           # Jobs to fill
param cost {PEOPLE, JOBS};          # Cost or preference rank
var Assign {PEOPLE, JOBS} binary;   # One if paired
minimize AssignmentCost:            # Total cost or rank of the chosen pairs
    sum {p in PEOPLE, j in JOBS} cost[p,j] * Assign[p,j];
subject to OneJob {p in PEOPLE}:    # Each person gets exactly one job
    sum {j in JOBS} Assign[p,j] = 1;
subject to OnePerson {j in JOBS}:   # Each job gets exactly one person
    sum {p in PEOPLE} Assign[p,j] = 1;

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PEOPLE"] = ["Coullard", "Daskin", "Hazen"]
ampl.set["JOBS"] = ["C118", "C138", "D237"]
ampl.param["cost"] = {
    ("Coullard", "C118"): 6, ("Coullard", "C138"): 9, ("Coullard", "D237"): 4,
    ("Daskin", "C118"): 11, ("Daskin", "C138"): 8, ("Daskin", "D237"): 1,
    ("Hazen", "C118"): 9, ("Hazen", "C138"): 10, ("Hazen", "D237"): 2,
}

In [5]:
# run.py
ampl.solve(solver="highs")
print(ampl.solve_result, ampl.obj["AssignmentCost"].value())
for (p, j), v in ampl.var["Assign"].to_dict().items():
    if v > 0.5:
        print(p, j, ampl.param["cost"][p, j])

HiGHS 1.14.0: optimal solution; objective 16
4 simplex iterations
1 branching nodes
solved 16.0
Coullard C118 6
Daskin C138 8
Hazen D237 2


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set PEOPLE := Coullard Daskin Hazen;
set JOBS := C118 C138 D237;
param cost:  C118  C138  D237 :=
  Coullard     6     9     4
  Daskin      11     8     1
  Hazen        9    10     2 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display solve_result, AssignmentCost;
display {p in PEOPLE, j in JOBS: Assign[p,j] > 0.5} cost[p,j];

HiGHS 1.14.0: optimal solution; objective 16
4 simplex iterations
1 branching nodes
solve_result = solved
AssignmentCost = 16

cost[p,j] :=
Coullard C118   6
Daskin   C138   8
Hazen    D237   2
;

